# 🏭 AI-Enabled Assets Performance & Predictive Maintenance Platform

This notebook allows you to run the full **Industrial Risk AI** platform directly in Google Colab.


### Step 1: 🚀 Setup Environment
Run this cell to install the platform and dependencies.

In [ ]:
import os
import subprocess
import time
import sys

# 1. Clone Repository
REPO_URL = "https://github.com/lmudu2/industrial-risk-ai.git"
REPO_DIR = "industrial-risk-ai"

if os.path.exists(REPO_DIR):
    !rm -rf {REPO_DIR}

print(f"Cloning {REPO_URL}...")
!git clone {REPO_URL}
os.chdir(f"/content/{REPO_DIR}")

# 2. Install Dependencies
print("Installing dependencies (this may take 2 minutes)...")
!pip install -v -r requirements.txt

import streamlit
print(f"\n✅ Installed Streamlit version: {streamlit.__version__}")
if streamlit.__version__ < "1.34.0":
    print("❌ ERROR: Streamlit version is too low for this project.")
    print("👉 GO TO MENU: 'Runtime' -> 'Restart Session' and run this cell again!")
else:
    print("✅ Version check passed!")

### Step 2: 📊 Generate Industrial Database
Run this cell to generate the synthesized SQLite database.

In [ ]:
if not os.path.exists("backend/eam_database.db"):
    print("Generating real-world industrial data (3-5 minutes)...")
    !python data/generate_data.py
else:
    print("✅ Database already exists.")

### Step 3: ⚡ Start & Monitor Logs
This cell starts the platform and displays **Live Error Logs** below.

In [ ]:
# --- 3. START PLATFORM ---
import subprocess, time, socket, os
from google.colab import output

def wait_for_port(port, timeout=30):
    start_time = time.time()
    while time.time() - start_time < timeout:
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
            if s.connect_ex(("localhost", port)) == 0: return True
        time.sleep(1)
    return False

print("🚀 Launching Industrial Risk AI Platform...")

# 1. Kill old sessions
!fuser -k 8501/tcp
!fuser -k 8000/tcp

# 2. Start Backend (Port 8000)
print("Starting Backend Service...")
subprocess.Popen(["uvicorn", "backend.main:app", "--host", "0.0.0.0", "--port", "8000"])

# 3. Start Frontend (Port 8501)
print("Starting Dashboard Service...")
# Using Native Colab Proxy flags (CORS disabled, Headless enabled)
subprocess.Popen(["streamlit", "run", "frontend/app.py", 
                  "--server.port", "8501", 
                  "--server.enableCORS", "False", 
                  "--server.headless", "True"])

if wait_for_port(8501):
    print("\n" + "="*60)
    print("✅ DASHBOARD READY!")
    print("Click the link below to open the secure dashboard:")
    print("="*60)
    # Native Colab Proxy Link
    output.serve_kernel_port_as_iframe(8501) # Optional: show inline
    print(f"\n🔗 Access Link: ")
    display(output.eval_js(f"google.colab.kernel.proxyPort(8501)"))
    print("\n" + "="*60)
else:
    print("\n❌ Error: Dashboard failed to start. Check dependencies in Step 1.")
